In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from config.setting import NUM_COLUMNS
from utils.helper import get_data_path
import math

# configurations
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
sns.set_theme(style="darkgrid")

plt.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

TARGET_COL = "price"  # target column name

df = pd.read_csv(get_data_path("final_data.csv"))

In [ ]:
print(df.columns)

In [ ]:
print(df.shape)

In [ ]:
print(df.info())

In [2]:
print(df.isna().sum())

address                  0
area                     0
house_direction      15638
balcony_direction    19117
floors                4306
bedrooms              1211
bathrooms             4331
legal_status          2934
furniture_state      10010
price                    0
year                     0
property_type            0
property_feature         0
lat                      0
lng                      0
dtype: int64


In [ ]:
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()
print("Number of duplicate rows:", num_duplicates)

In [ ]:
df = df.drop_duplicates()


In [ ]:
print(len(df))

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

In [ ]:
df = df[(df.price > df.price.quantile(0.01)) &
        (df.price < df.price.quantile(0.99))]

In [ ]:
print("Target column:", TARGET_COL)
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)
print(df[num_cols].describe().T)

In [ ]:
safe_cat_cols = [c for c in cat_cols if df[c].nunique() < 50]
for col in safe_cat_cols:
    plt.figure(figsize=(14, 6))
    sns.countplot(x=col, data=df)
    plt.xticks(rotation=45)
    plt.title(f"Distribution of {col}")
    plt.show()

df = df[(df.price > df.price.quantile(0.01)) &
        (df.price < df.price.quantile(0.99))]

In [ ]:

n_cols = 3
n_rows = math.ceil(len(num_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(col, fontsize=8)
for col in num_cols:
    print(col)
    print("Top 5 max values:")
    print(df[col].sort_values(ascending=False).head())
    print()
# Ẩn subplot dư
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()



In [ ]:
df = df[df["area"] < df["area"].quantile(0.99)]
df["area"].sort_values(ascending=False).head()

In [ ]:
# identify presence of highly correlated columns & feature relationships
plt.figure(figsize=(10, 5))
sns.heatmap(
    df[num_cols].corr(),
    annot=True,
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Heatmap")
plt.show()

# Correlation with target
corr_with_target = df[num_cols].corr()[TARGET_COL].sort_values(ascending=False)
print("\nCorrelation with target:")
print(corr_with_target)


In [ ]:
# target column distribution
plt.figure(figsize=(6, 4))
sns.histplot(df['price'], bins=40, kde=True)
plt.title("Target Distribution: Price House Value")
plt.xlabel("Price House Value")
plt.show()
print(df['price'].value_counts())


In [ ]:
print("Floors > 15:", (df["floors"] > 15).sum())
print("Floors is NaN:", df["floors"].isna().sum())